# Experiment: UHI Local Run 1 (Refined, Numbered Steps)

Objective:
- Run the Bangalore UHI pipeline in a clean, reproducible order.
- Remove fragile notebook patterns from the previous run.
- Add explicit verification checks for each stage.
- Follow numbered steps below to track progress.

Deliverables:
- Preprocessed sensor tiles
- Aligned per-date stacks (`processed/stacks/aligned`)
- Seasonal stacks (`processed/stacks/seasonal`)
- Validation outputs and pilot model artifacts (optional)


## Numbered Run Plan

Execute top-to-bottom in this exact order.

1. **Step 0: Environment Helpers** - utilities cell (`run_cmd`, `ensure_dir`).
2. **Step 1: Repo Sync** - clone/pull `octo` in Colab runtime.
3. **Step 2: Mount Drive + Paths** - set canonical directories.
4. **Step 3 (Optional): Raw Data Sync** - copy raw exports if `UHI_Project/raw_data` is incomplete.
5. **Step 4: Preprocess** - generate Landsat/Sentinel tiles.
6. **Step 5: Preprocess Verification** - tile counts + corruption check.
7. **Step 6: Alignment** - run per-date and seasonal alignment.
8. **Step 7: Seasonal/Feature Validation** - physical ranges + assertion checks.
9. **Step 8 (Optional): Modeling** - pilot dataset, XGBoost, SHAP, visualization.
10. **Step 9 (Optional): MODIS Validation** - external validation artifacts.
11. **Step 10: Final Rubric** - interpret outcomes against pass/fail criteria.

If you pause midway, restart from the latest **required** step above, then continue sequentially.


In [ ]:
# Step 0: Environment helpers and command runner

from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Iterable


def run_cmd(cmd: list[str], cwd: str | Path | None = None, check: bool = True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result


def ensure_dir(path: str | Path):
    Path(path).mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])


In [ ]:
# Step 1: Repo sync (idempotent)
REPO_DIR = Path("/content/octo")
REPO_URL = "https://github.com/githubbermoon/octo.git"

if REPO_DIR.exists():
    run_cmd(["git", "pull"], cwd=REPO_DIR)
else:
    run_cmd(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


In [ ]:
# Step 2: Drive mount and canonical paths
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

BASE_DIR = Path("/content/drive/MyDrive/UHI_Project")
RAW_DIR = BASE_DIR / "raw_data"
PROC_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
VALIDATION_DIR = BASE_DIR / "validation"

for p in [
    RAW_DIR,
    PROC_DIR / "landsat8",
    PROC_DIR / "landsat9",
    PROC_DIR / "sentinel2",
    PROC_DIR / "stacks",
    MODELS_DIR,
    VALIDATION_DIR,
]:
    ensure_dir(p)

print("Base:", BASE_DIR)


## Step 3 (Optional): Sync Raw Exports into `UHI_Project/raw_data`

Use this only if `raw_data/*` is missing.


In [ ]:
# Step 3 Code: optional raw data sync

SYNC_RAW = False  # set True if you need to copy from Research_Data_Seasonal_Scenes
SOURCE = Path("/content/drive/MyDrive/Research_Data_Seasonal_Scenes")
SENSORS = ["Sentinel-2", "Landsat-8", "Landsat-9", "MODIS"]

if SYNC_RAW:
    for sensor in SENSORS:
        src = SOURCE / sensor
        dst = RAW_DIR / sensor
        ensure_dir(dst)
        if src.exists():
            # incremental copy; safer than blind full copy in long-running cells
            run_cmd(["rsync", "-ah", "--ignore-existing", f"{src}/", f"{dst}/"])
        else:
            print(f"[warn] missing source: {src}")
else:
    print("SYNC_RAW=False; skipping raw copy.")


## Step 4: Preprocess (Must-Pass Stage)

Expected checks from a healthy run:
- Landsat-8 tiles: >= 100
- Landsat-9 tiles: >= 60
- Sentinel-2 tiles: >= 3000
- Corrupt `.npz` count: 0

> Note: Preprocessing is already completed for your dataset. This step is intentionally disabled by default.
> Keep it OFF unless raw rasters change.


In [ ]:
# Step 4 Code: run preprocessing scripts in deterministic order
# NOTE: This stage is disabled by default because preprocessing is already complete.
# Only set RUN_PREPROCESS=True if raw raster inputs changed or you want a full rebuild.

RUN_PREPROCESS = False  # disabled: preprocessing already completed

if RUN_PREPROCESS:
    run_cmd([
        "python", "scripts/preprocess.py",
        "--input", str(RAW_DIR / "Landsat-8"),
        "--output", str(PROC_DIR / "landsat8"),
        "--config", "configs/configD1.yaml",
    ])

    run_cmd([
        "python", "scripts/preprocess.py",
        "--input", str(RAW_DIR / "Landsat-9"),
        "--output", str(PROC_DIR / "landsat9"),
        "--config", "configs/configD1.yaml",
    ])

    run_cmd([
        "python", "scripts/preprocess.py",
        "--input", str(RAW_DIR / "Sentinel-2"),
        "--output", str(PROC_DIR / "sentinel2"),
        "--config", "configs/configD1.yaml",
    ])
else:
    print("RUN_PREPROCESS=False; skipping. Preprocessing intentionally skipped because tiles already exist.")


In [ ]:
# Step 5: preprocess verification (counts + corruption guard)
import numpy as np

l8_tiles = sorted((PROC_DIR / "landsat8" / "tiles").glob("*.npz"))
l9_tiles = sorted((PROC_DIR / "landsat9" / "tiles").glob("*.npz"))
s2_tiles = sorted((PROC_DIR / "sentinel2" / "tiles").glob("*.npz"))

print("Landsat-8 tiles:", len(l8_tiles))
print("Landsat-9 tiles:", len(l9_tiles))
print("Sentinel-2 tiles:", len(s2_tiles))

# Soft expectations (based on your previous successful run)
assert len(l8_tiles) >= 100, "Landsat-8 tile count unexpectedly low"
assert len(l9_tiles) >= 60, "Landsat-9 tile count unexpectedly low"
assert len(s2_tiles) >= 3000, "Sentinel-2 tile count unexpectedly low"

bad = []
for p in l8_tiles[:80] + l9_tiles[:80] + s2_tiles[:80]:
    try:
        _ = np.load(p, allow_pickle=True)
    except Exception as e:
        bad.append((str(p), str(e)))

print("Sample corruption check failures:", len(bad))
if bad:
    print("First bad file:", bad[0])


## Step 6: Alignment and Seasonal Stack Generation


In [ ]:
# Step 6A: per-date alignment (pilot path used by pilot_plan scripts)
RUN_ALIGN_FEATURES = True
ALIGN_LIMIT = 0  # set >0 for quick debug

if RUN_ALIGN_FEATURES:
    cmd = [
        "python", "scripts/align_features.py",
        "--config", "configs/configD1.yaml",
    ]
    if ALIGN_LIMIT > 0:
        cmd += ["--limit", str(ALIGN_LIMIT)]
    run_cmd(cmd)
else:
    print("RUN_ALIGN_FEATURES=False; skipping.")


In [ ]:
# Step 6B: seasonal alignment (preferred scientific path)
RUN_ALIGN_SEASONAL = True
WORLDCOVER = RAW_DIR / "worldcover" / "worldcover_2021.tif"

if RUN_ALIGN_SEASONAL:
    cmd = ["python", "scripts/align_seasonal.py", "--config", "configs/configD1.yaml"]
    if WORLDCOVER.exists():
        cmd += ["--worldcover", str(WORLDCOVER)]
    run_cmd(cmd)
else:
    print("RUN_ALIGN_SEASONAL=False; skipping.")


In [ ]:
# Step 7A: seasonal verification with explicit expected ranges
import numpy as np

seasonal_files = sorted((PROC_DIR / "stacks" / "seasonal").glob("*.npz"))
print("Seasonal file count:", len(seasonal_files))
assert len(seasonal_files) == 12, "Expected 12 seasonal files (2020-2025 x summer/winter)"

d = np.load(seasonal_files[0], allow_pickle=True)
ndvi = d["ndvi"]
ndbi = d["ndbi"]
lst = d["lst"]
n_obs = d["n_obs_lst"] if "n_obs_lst" in d.files else d.get("n_obs")

print(f"NDVI range: [{np.nanmin(ndvi):.3f}, {np.nanmax(ndvi):.3f}]")
print(f"NDBI range: [{np.nanmin(ndbi):.3f}, {np.nanmax(ndbi):.3f}]")
print(f"LST range:  [{np.nanmin(lst):.1f}K, {np.nanmax(lst):.1f}K]")

# Expected envelope from your previous successful run
assert np.nanmin(ndvi) >= -1.05 and np.nanmax(ndvi) <= 1.05, "NDVI out of physical range"
assert np.nanmin(ndbi) >= -1.05 and np.nanmax(ndbi) <= 1.05, "NDBI out of physical range"
assert 295 <= np.nanmin(lst) <= 310 and 320 <= np.nanmax(lst) <= 330, "LST range abnormal"


In [ ]:
# Step 7B: assertion-grade feature validation
import sys
import numpy as np

sys.path.insert(0, str(REPO_DIR))
from scripts.validate_features import run_all_validations

d = np.load(sorted((PROC_DIR / "stacks" / "seasonal").glob("*.npz"))[0], allow_pickle=True)
res = run_all_validations({"ndvi": d["ndvi"], "ndbi": d["ndbi"], "lst": d["lst"]}, season="summer")
print("All valid:", res.get("all_valid"))
assert bool(res.get("all_valid")), "Feature validation did not pass"


## Step 8 (Optional): Modeling and Explainability (Pilot)

Use this to reproduce your pilot metrics quickly. Keep in mind this is still pilot-grade, not final paper-grade.


In [ ]:
# Step 8 Code: optional modeling + explainability pipeline

RUN_MODELING = False  # set True to run pilot model training

if RUN_MODELING:
    run_cmd([
        "python", "pilot_plan/01_prepare_data.py",
        "--aligned-dir", str(PROC_DIR / "stacks" / "aligned"),
        "--worldcover", str(WORLDCOVER),
        "--output", str(BASE_DIR / "pilot_train.csv"),
    ])

    run_cmd([
        "python", "pilot_plan/02_train_xgb.py",
        "--data", str(BASE_DIR / "pilot_train.csv"),
        "--output", str(MODELS_DIR),
    ])

    run_cmd([
        "python", "pilot_plan/03_shap_analysis.py",
        "--model", str(MODELS_DIR / "xgb_pilot.json"),
        "--data", str(BASE_DIR / "pilot_train.csv"),
        "--output", str(BASE_DIR / "figures"),
    ])

    run_cmd([
        "python", "pilot_plan/04_visualize_uhi.py",
        "--model", str(MODELS_DIR / "xgb_pilot.json"),
        "--aligned-dir", str(PROC_DIR / "stacks" / "aligned"),
        "--output", str(BASE_DIR / "figures"),
    ])
else:
    print("RUN_MODELING=False; skipping model stage.")


In [ ]:
# Step 9: optional MODIS validation stage
RUN_MODIS_VALIDATION = False  # set True after seasonal outputs exist

if RUN_MODIS_VALIDATION:
    run_cmd([
        "python", "scripts/validate_modis.py",
        "--config", "configs/configD1.yaml",
    ])

    pix = VALIDATION_DIR / "pixel_wise" / "level1_pixel_metrics.json"
    reg = VALIDATION_DIR / "regional" / "level2_regional_metrics.json"

    if pix.exists():
        print("Pixel metrics:")
        print(json.dumps(json.loads(pix.read_text()), indent=2)[:2000])
    if reg.exists():
        print("Regional metrics:")
        print(json.dumps(json.loads(reg.read_text()), indent=2)[:2000])
else:
    print("RUN_MODIS_VALIDATION=False; skipping.")


## Step 10: Verification Rubric (What Results Should Look Like)

Use this as a go/no-go reference:

| Stage | Good signal | Needs attention |
|---|---|---|
| Preprocess | No crashes; tile counts roughly L8>=100, L9>=60, S2>=3000; no corrupt NPZ files | KeyboardInterrupt, EOFError, low tile counts |
| Seasonal stacks | Exactly 12 files; NDVI/NDBI within [-1,1]; LST approx 301-325K | Missing seasons, extreme NDVI/NDBI or LST |
| Physical consistency | `corr(NDVI,LST)` negative, `corr(NDBI,LST)` positive | Opposite signs or near-zero meaningless correlations |
| XGBoost pilot | Validation RMSE around 3-4C, R2 around 0.5+ | RMSE much higher, unstable feature importances |
| UHI seasonal summary | Urban hotter than vegetation/rural by ~1-3K | Urban not hotter, inconsistent seasonality |

Important:
- `validate_modis.py` Level-1 can look poor if run with placeholder-baseline paths; interpret only after true model inference is wired.
- `pilot_plan/04_visualize_uhi.py` is explicitly pilot-grade (heuristic masks), not final reporting.


## Step 11: Definition of Done (Publication Gate)

This gate tells you how close the project is to the publication objective.

Pass conditions:
- Data integrity: tile counts and corruption checks pass.
- Seasonal artifacts: 12 seasonal files exist.
- Physical behavior: NDVI-LST negative, NDBI-LST positive.
- Modeling baseline: XGBoost metrics meet target thresholds (if modeling was run).
- Validation artifacts: regional/pixel validation JSON exists (after validation stage).

Interpretation:
- **Green (>=80%)**: ready for manuscript draft and figure freeze.
- **Yellow (60-79%)**: technically solid but missing key validation/model gates.
- **Red (<60%)**: not publication-ready.


In [ ]:
# Step 11 Code: objective completion scorecard
import json
import numpy as np
from pathlib import Path

checks = []

def add_check(name, ok, detail):
    checks.append({"name": name, "ok": bool(ok), "detail": detail})

# 1) Tile counts
l8 = len(list((PROC_DIR / "landsat8" / "tiles").glob("*.npz")))
l9 = len(list((PROC_DIR / "landsat9" / "tiles").glob("*.npz")))
s2 = len(list((PROC_DIR / "sentinel2" / "tiles").glob("*.npz")))
add_check("Tile counts", (l8 >= 100 and l9 >= 60 and s2 >= 3000), f"L8={l8}, L9={l9}, S2={s2}")

# 2) Seasonal artifacts
seasonal_files = sorted((PROC_DIR / "stacks" / "seasonal").glob("*.npz"))
add_check("Seasonal file count", len(seasonal_files) == 12, f"count={len(seasonal_files)}")

# 3) Physical consistency on first seasonal file
if seasonal_files:
    d = np.load(seasonal_files[0], allow_pickle=True)
    ndvi = d["ndvi"].ravel()
    ndbi = d["ndbi"].ravel()
    lst = d["lst"].ravel()
    m1 = np.isfinite(ndvi) & np.isfinite(lst)
    m2 = np.isfinite(ndbi) & np.isfinite(lst)
    c1 = float(np.corrcoef(ndvi[m1], lst[m1])[0, 1]) if m1.sum() > 100 else np.nan
    c2 = float(np.corrcoef(ndbi[m2], lst[m2])[0, 1]) if m2.sum() > 100 else np.nan
    add_check("Physical consistency", (c1 < 0 and c2 > 0), f"corr(NDVI,LST)={c1:.3f}, corr(NDBI,LST)={c2:.3f}")
else:
    add_check("Physical consistency", False, "No seasonal files")

# 4) XGBoost pilot metrics (optional if modeling not run)
metrics_path = MODELS_DIR / "xgb_pilot_metrics.json"
if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    rmse = m.get("val_rmse", m.get("rmse", np.nan))
    r2 = m.get("val_r2", m.get("r2", np.nan))
    ok = (isinstance(rmse, (int, float)) and isinstance(r2, (int, float)) and rmse <= 4.0 and r2 >= 0.50)
    add_check("XGBoost pilot target", ok, f"val_rmse={rmse}, val_r2={r2}")
else:
    add_check("XGBoost pilot target", False, "metrics file missing (run Step 8)")

# 5) Validation artifact presence
pix = VALIDATION_DIR / "pixel_wise" / "level1_pixel_metrics.json"
reg = VALIDATION_DIR / "regional" / "level2_regional_metrics.json"
add_check("Validation artifacts", (pix.exists() and reg.exists()), f"pixel={pix.exists()}, regional={reg.exists()}")

# Summary
total = len(checks)
passed = sum(1 for c in checks if c["ok"])
score = 100.0 * passed / total if total else 0.0

print("="*64)
print("OBJECTIVE COMPLETION SCORECARD")
print("="*64)
for c in checks:
    status = "PASS" if c["ok"] else "FAIL"
    print(f"[{status}] {c['name']}: {c['detail']}")
print("-"*64)
print(f"Score: {passed}/{total} = {score:.1f}%")
if score >= 80:
    print("Status: GREEN (publication drafting ready)")
elif score >= 60:
    print("Status: YELLOW (close, but key gates missing)")
else:
    print("Status: RED (not publication-ready yet)")


## Step 12: Drift and Misalignment Check vs Published Ranges

This section compares your generated UHI signals against commonly reported Bengaluru ranges in peer-reviewed studies.

Benchmark references used here:
- Average UHI in Bengaluru reported around **1.9 C** (city-scale analysis).
- Seasonal/annual Bengaluru UHI often observed in roughly **0.6 to 2.6 C** envelopes depending on zone and period.

Interpretation:
- If your seasonal UHI means mostly fall within ~0.6-2.6 C and preserve expected signs, your outputs are broadly plausible.
- Large deviations do not automatically mean failure, but require audit of preprocessing, masks, and validation protocol.


In [ ]:
# Step 12 Code: literature-range comparison for UHI drift
import numpy as np
from pathlib import Path

seasonal_files = sorted((PROC_DIR / "stacks" / "seasonal").glob("*.npz"))
if not seasonal_files:
    raise RuntimeError("No seasonal files found. Run Step 6B first.")

# Published envelopes (Bengaluru-focused studies)
BENCH_AVG_UHI = 1.9
BENCH_RANGE_LOW = 0.6
BENCH_RANGE_HIGH = 2.6

rows = []
for f in seasonal_files:
    d = np.load(f, allow_pickle=True)
    lst = d["lst"]

    # Prefer explicit masks if present
    if "is_urban" in d.files and "is_vegetation" in d.files:
        urban_mask = d["is_urban"] == 1
        rural_mask = d["is_vegetation"] == 1
    else:
        # fallback heuristic
        ndvi = d["ndvi"]
        ndbi = d["ndbi"]
        urban_mask = (ndbi > 0.1) & (ndvi < 0.2)
        rural_mask = (ndvi > 0.35)

    u = lst[urban_mask & np.isfinite(lst)]
    r = lst[rural_mask & np.isfinite(lst)]
    if len(u) == 0 or len(r) == 0:
        continue

    u_mean = float(np.nanmean(u))
    r_mean = float(np.nanmean(r))
    uhi = u_mean - r_mean

    rows.append((f.stem, uhi, u_mean, r_mean, len(u), len(r)))

if not rows:
    raise RuntimeError("Could not compute UHI from seasonal stacks; check masks.")

uhis = np.array([x[1] for x in rows], dtype=float)
mean_uhi = float(np.mean(uhis))
min_uhi = float(np.min(uhis))
max_uhi = float(np.max(uhis))
within = float(np.mean((uhis >= BENCH_RANGE_LOW) & (uhis <= BENCH_RANGE_HIGH)) * 100)

print("="*72)
print("DRIFT CHECK VS PUBLISHED BENGALURU UHI ENVELOPE")
print("="*72)
for name, uhi, u_mean, r_mean, nu, nr in rows:
    print(f"{name}: UHI={uhi:.2f}K | urban={u_mean:.1f}K (n={nu}) | rural={r_mean:.1f}K (n={nr})")

print("-"*72)
print(f"Your seasonal UHI summary: mean={mean_uhi:.2f}K, min={min_uhi:.2f}K, max={max_uhi:.2f}K")
print(f"Published reference: avg~{BENCH_AVG_UHI:.1f}K, envelope~[{BENCH_RANGE_LOW:.1f}, {BENCH_RANGE_HIGH:.1f}]K")
print(f"Seasons within envelope: {within:.1f}%")

if within >= 80 and 0.8 <= mean_uhi <= 2.8:
    print("Status: CONSISTENT (no major drift detected)")
elif within >= 50:
    print("Status: BORDERLINE (some drift; review masks/coverage)")
else:
    print("Status: DRIFT SUSPECTED (audit preprocessing/alignment/validation)")
